# 07 · Exploratory Biological Annotation

**Goal**: Explore the biological characteristics of Leiden clusters by scoring curated candidate tissue-specific gene programs.

**Input**: results/adata/06_markers.h5ad
**Output**: results/adata/07_annotated.h5ad, annotation evidence tables, and figures.

**Note**: Biological annotations are exploratory and do not represent validated cell identities. The 56 µm spatial bins may contain multiple cells, and candidate gene programs require further biological validation.

## 1. Imports and paths

In [ ]:
from pathlib import Path
from importlib.metadata import version
import sys
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse
import matplotlib.pyplot as plt


def setup_annotation_paths():
    root = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "results/adata/06_markers.h5ad").exists()), None)
    if root is None:
        raise FileNotFoundError("Run Step 06 first: results/adata/06_markers.h5ad not found")
    input_path = root / "results/adata/06_markers.h5ad"
    output_path = root / "results/adata/07_annotated.h5ad"
    fig_dir = root / "results/figures"
    table_dir = root / "results/tables"
    for path in (output_path.parent, fig_dir, table_dir):
        path.mkdir(parents=True, exist_ok=True)
    return root, input_path, output_path, fig_dir, table_dir


ROOT, INPUT, OUTPUT, FIG, TAB = setup_annotation_paths()
SEED = 42
np.random.seed(SEED)
print("Python:", sys.executable)
print("Scanpy:", version("scanpy"), "AnnData:", version("anndata"))
print("Project root:", ROOT)

## 2. Load and validate marker analysis results

Load the AnnData output from Step 06 and verify that Leiden cluster labels, marker-ranking results, raw counts, and spatial/expression neighbor graphs are available. Preserve the original Leiden cluster IDs for downstream biological annotation. All annotations remain exploratory and do not represent validated cell-type identities.

In [ ]:
def load_and_validate_markers(input_path, key):
    adata = sc.read_h5ad(input_path)
    assert "leiden" in adata.obs and "spatial" in adata.obsm
    assert key in adata.uns, "Run marker analysis (Step 06) first"
    assert "counts" in adata.layers and sparse.issparse(adata.layers["counts"])
    assert "spatial_connectivities" in adata.obsp and "connectivities" in adata.obsp
    assert "log1p" in adata.uns
    adata.obs["leiden"] = adata.obs["leiden"].astype("category")
    clusters = list(adata.obs["leiden"].cat.categories)
    return adata, clusters


KEY = "rank_genes_leiden_wilcoxon"
adata, clusters = load_and_validate_markers(INPUT, KEY)
print("AnnData:", adata.shape, "Leiden clusters:", len(clusters))

## 3. Candidate biological programs and gene coverage

Define candidate gene signatures for cartilage, bone, stromal, vascular, immune, and other tissue-associated programs. Check gene availability and export the coverage summary (07_program_gene_coverage.csv).

Missing genes are reported and excluded from scoring. These signatures are exploratory, not validated cell-type classifiers. Gene coverage does not indicate biological activity.

In [ ]:
PROGRAMS = {
    "Cartilage / chondrocyte-like": ["COL2A1", "ACAN", "SOX9", "COMP", "COL9A1", "COL11A1"],
    "Bone matrix / osteogenic-like": ["COL1A1", "COL1A2", "SPP1", "IBSP", "BGLAP", "SP7"],
    "Fibroblast / stromal-like": ["DCN", "LUM", "COL3A1", "PDGFRA", "THY1", "FBLN1"],
    "Vascular endothelial-like": ["PECAM1", "VWF", "KDR", "EMCN", "CDH5", "ESAM"],
    "Perivascular / smooth-muscle-like": ["ACTA2", "TAGLN", "MYH11", "RGS5", "PDGFRB", "MCAM"],
    "Myeloid / macrophage-like": ["LST1", "TYROBP", "C1QA", "C1QB", "CD68", "CSF1R"],
    "Lymphoid-like": ["PTPRC", "CD3D", "CD3E", "TRAC", "MS4A1", "CD79A"],
    "Adipocyte-like": ["ADIPOQ", "PLIN1", "FABP4", "LPL", "LEP", "CIDEC"],
    "Erythroid-like": ["HBB", "HBA1", "HBA2", "ALAS2", "GYPA", "SLC4A1"],
}


def calculate_program_coverage(adata, programs, table_dir):
    coverage_rows = []
    found = {}
    for program, genes in programs.items():
        present = [g for g in genes if g in adata.var_names]
        missing = [g for g in genes if g not in adata.var_names]
        found[program] = present
        coverage_rows.append({"program": program, "n_reference_genes": len(genes),
                              "n_detected_genes": len(present),
                              "coverage_fraction": len(present) / len(genes),
                              "present_genes": ";".join(present),
                              "missing_genes": ";".join(missing)})
    coverage = pd.DataFrame(coverage_rows)
    coverage.to_csv(table_dir / "07_program_gene_coverage.csv", index=False)
    assert any(len(g) >= 2 for g in found.values()), "Too few candidate genes in the dataset"
    return coverage, found


coverage, found = calculate_program_coverage(adata, PROGRAMS, TAB)
display(coverage)

## 4. Score candidate biological programs by cluster

Calculate the mean log1p-normalized expression of candidate genes for each bin and aggregate by Leiden cluster. Convert cluster-level means into z-scores across clusters and export the results as CSV tables.

Only programs with at least two available genes are scored. Scores are exploratory and indicate relative expression patterns, not validated cell-type identities.

In [ ]:
def score_programs_by_cluster(adata, found, clusters, table_dir):
    valid = {name: genes for name, genes in found.items() if len(genes) >= 2}
    program_names = list(valid)
    # Subset only the small marker union to avoid densifying the full matrix.
    union = list(dict.fromkeys(g for genes in valid.values() for g in genes))
    sub = adata[:, union].X
    small = sub.toarray() if sparse.issparse(sub) else np.asarray(sub)
    idx = {gene: i for i, gene in enumerate(union)}
    per_bin = pd.DataFrame(index=adata.obs_names)
    for name, genes in valid.items():
        per_bin[name] = small[:, [idx[g] for g in genes]].mean(axis=1)
    per_bin["leiden"] = adata.obs["leiden"].astype(str).to_numpy()
    means = per_bin.groupby("leiden", sort=False)[program_names].mean()
    means = means.reindex([str(c) for c in clusters])
    std = means.std(axis=0, ddof=0)
    z = (means - means.mean(axis=0)) / std.replace(0, np.nan)
    z = z.fillna(0.0)
    means.to_csv(table_dir / "07_cluster_program_mean_expression.csv", index_label="leiden")
    z.to_csv(table_dir / "07_cluster_program_zscores.csv", index_label="leiden")
    return valid, program_names, means, z


valid, program_names, means, z = score_programs_by_cluster(adata, found, clusters, TAB)
print("Programs scored:", len(program_names), "Clusters:", len(means))
display(z.round(2).head())

## 5. Evidence from Step 06 marker rankings

Compare each candidate biological program with the top 100 positive marker genes of each Leiden cluster from Step 06. Count overlapping genes and export the results (07_program_marker_evidence.csv).

Marker overlap provides supporting exploratory evidence, not independent biological validation or confirmed cell-type identities.

In [ ]:
def collect_marker_evidence(adata, clusters, valid, key, table_dir):
    marker_rows = []
    for c in clusters:
        df = sc.get.rank_genes_groups_df(adata, group=str(c), key=key)
        df = df.loc[df["logfoldchanges"].fillna(-np.inf) > 0].sort_values("scores", ascending=False)
        top100 = set(df.head(100)["names"].astype(str))
        for name, genes in valid.items():
            hits = [g for g in genes if g in top100]
            marker_rows.append({"leiden": str(c), "program": name,
                                "n_signature_genes": len(genes),
                                "n_top100_marker_hits": len(hits),
                                "top100_marker_hits": ";".join(hits)})
    evidence = pd.DataFrame(marker_rows)
    evidence.to_csv(table_dir / "07_program_marker_evidence.csv", index=False)
    return evidence


evidence = collect_marker_evidence(adata, clusters, valid, KEY, TAB)
print("Evidence rows:", len(evidence))

## 6. Conservative candidate labels and review flags

Propose a biological program label for each Leiden cluster only when the top program has a z-score ≥ 1.0, at least two matching top-100 marker genes, and a z-score margin ≥ 0.5 over the runner-up.

Otherwise, label the cluster as "Unresolved / mixed". Export the provisional annotations (07_provisional_cluster_annotations.csv). All labels are exploratory and require manual review.

In [ ]:
def propose_cluster_annotations(adata, z, evidence, table_dir,
                                min_z=1.0, min_hits=2, min_margin=0.5):
    annotation_rows = []
    for c in z.index:
        ordered = z.loc[c].sort_values(ascending=False)
        best = str(ordered.index[0]); best_z = float(ordered.iloc[0])
        second_z = float(ordered.iloc[1]) if len(ordered) > 1 else float("nan")
        margin = best_z - second_z if np.isfinite(second_z) else 0.0
        match = evidence.loc[(evidence["leiden"] == c) & (evidence["program"] == best)].iloc[0]
        hits = int(match["n_top100_marker_hits"])
        proposed = best_z >= min_z and hits >= min_hits and margin >= min_margin
        annotation_rows.append({"leiden": c, "n_bins": int((adata.obs["leiden"].astype(str) == c).sum()),
                                "candidate_program": best,
                                "best_program_zscore": best_z,
                                "runner_up_program": str(ordered.index[1]) if len(ordered) > 1 else "",
                                "zscore_margin": margin,
                                "n_top100_marker_hits": hits,
                                "supporting_markers": str(match["top100_marker_hits"]),
                                "provisional_label": best if proposed else "Unresolved / mixed",
                                "review_status": "Manual review required",
                                "interpretation": "Spatial-bin program; not a cell-type call"})
    annotation = pd.DataFrame(annotation_rows)
    annotation.to_csv(table_dir / "07_provisional_cluster_annotations.csv", index=False)
    return annotation


MIN_Z = 1.0
MIN_HITS = 2
MIN_MARGIN = 0.5
annotation = propose_cluster_annotations(adata, z, evidence, TAB, MIN_Z, MIN_HITS, MIN_MARGIN)
print("Proposed candidates:", (annotation["provisional_label"] != "Unresolved / mixed").sum(),
      "/", len(annotation))
display(annotation[["leiden", "candidate_program", "best_program_zscore",
                    "n_top100_marker_hits", "provisional_label"]])

## 7. Visualize program patterns and spatial distribution

Visualize relative biological program z-scores across Leiden clusters using a heatmap. Map the spatial distribution of bins colored by their provisional biological program labels, with unresolved clusters shown in gray.

Export both figures as PNG files. Spatial plots use coordinates only, without tissue-image overlays. All program labels remain exploratory.

In [ ]:
def plot_program_score_heatmap(z, fig_dir):
    fig, ax = plt.subplots(figsize=(12, max(7, len(z) * 0.28)))
    im = ax.imshow(z.to_numpy(), aspect="auto", cmap="coolwarm", vmin=-2, vmax=2)
    ax.set_xticks(range(len(z.columns)), z.columns, rotation=60, ha="right", fontsize=8)
    ax.set_yticks(range(len(z.index)), z.index, fontsize=8)
    ax.set(xlabel="Candidate tissue program", ylabel="Leiden cluster",
           title="Exploratory program scores (z across clusters; clipped ±2)")
    fig.colorbar(im, ax=ax, label="Relative z-score", shrink=0.75)
    fig.tight_layout()
    fig.savefig(fig_dir / "07_program_score_heatmap.png", dpi=170, bbox_inches="tight")
    plt.show()


def plot_provisional_spatial_programs(adata, annotation, fig_dir):
    mapping = dict(zip(annotation["leiden"], annotation["provisional_label"]))
    adata.obs["provisional_domain_program"] = pd.Categorical(
        adata.obs["leiden"].astype(str).map(mapping).fillna("Unresolved / mixed"))
    xy = np.asarray(adata.obsm["spatial"])
    labels = adata.obs["provisional_domain_program"].astype(str).to_numpy()
    unique_labels = list(pd.unique(labels))
    colors = plt.get_cmap("tab20", max(1, len(unique_labels)))
    fig, ax = plt.subplots(figsize=(10, 8))
    for i, label in enumerate(unique_labels):
        mask = labels == label
        ax.scatter(xy[mask, 0], xy[mask, 1], s=3, linewidths=0,
                   color="0.7" if label == "Unresolved / mixed" else colors(i),
                   label=f"{label} ({mask.sum():,})", rasterized=True)
    ax.invert_yaxis()
    ax.set_aspect("equal", adjustable="box")
    ax.set(xlabel="Fullres x (pixels)", ylabel="Fullres y (pixels)",
           title="Provisional spatial-bin programs (no image overlay)")
    ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=8, markerscale=4)
    fig.tight_layout()
    fig.savefig(fig_dir / "07_provisional_program_spatial.png", dpi=170, bbox_inches="tight")
    plt.show()


plot_program_score_heatmap(z, FIG)
plot_provisional_spatial_programs(adata, annotation, FIG)

## 8. Save and reload validation

Save the annotated AnnData with processing settings and reload it to verify data integrity, preserved Leiden clusters, spatial graphs, raw counts, and output files.

Annotations remain provisional and do not represent validated cell-type identities.

In [ ]:
def save_and_verify_annotations(adata, output_path, table_dir, fig_dir,
                                key, min_z, min_hits, min_margin):
    adata.uns["step07_settings"] = {
        "annotation_type": "exploratory_spatial_bin_program",
        "score": "mean_log1p_expression_per_program_then_cluster_zscore",
        "min_program_zscore": float(min_z),
        "min_top100_marker_hits": int(min_hits),
        "min_zscore_margin": float(min_margin),
        "manual_review_required": True,
        "not_cell_type_labels": True,
        "qc_filters_applied": False,
        "image_registration_verified": False,
    }
    adata.write_h5ad(output_path, compression="gzip")
    check = sc.read_h5ad(output_path)
    assert check.shape == adata.shape
    assert check.obs["leiden"].equals(adata.obs["leiden"])
    assert "provisional_domain_program" in check.obs
    assert np.array_equal(check.obsm["spatial"], adata.obsm["spatial"])
    assert "spatial_connectivities" in check.obsp and "connectivities" in check.obsp
    assert "counts" in check.layers and sparse.issparse(check.layers["counts"])
    assert key in check.uns
    assert (table_dir / "07_provisional_cluster_annotations.csv").exists()
    assert (fig_dir / "07_program_score_heatmap.png").exists()
    return check


check = save_and_verify_annotations(adata, OUTPUT, TAB, FIG, KEY, MIN_Z, MIN_HITS, MIN_MARGIN)
print("PASS — 07_biological_annotation completed")
print("AnnData:", check.shape)
print("Leiden clusters:", check.obs["leiden"].nunique())
print("Candidate programs:", len(program_names))
print("Provisional candidate clusters:",
      int((annotation["provisional_label"] != "Unresolved / mixed").sum()))
print("Output:", OUTPUT.relative_to(ROOT))

## Next: 08. Robustness Checks

In `08_robustness_checks.ipynb`, we will assess the robustness of exploratory clustering and biological annotation results.

**Input**: results/adata/07_annotated.h5ad

**Note**: Biological annotations remain provisional and require manual validation.